# Phase 2: Classification Fine-Tuning

**Goal:** Fine-tune the pretrained TimeSformer encoder to classify threshold typologies.

**What this does:**
- Loads pretrained encoder from MAE
- Adds classification head (8 typologies)
- Fine-tunes on labeled data
- **Result: Embeddings will cluster by typology!**

---

## Why This Step is Critical

After MAE pretraining:
- ✓ Encoder learned spatial patterns
- ✗ Embeddings don't cluster by typology (no supervision)

After classification fine-tuning:
- ✓ Encoder learns typology-specific features
- ✓ Embeddings WILL cluster by typology
- ✓ Model can predict threshold types

## Setup

In [1]:
import os
import sys
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import time
from datetime import datetime
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

# Adjust path if running from model folder
if os.path.basename(os.getcwd()) == 'model':
    os.chdir('..')

from config import get_mae_config, get_small_config
from timesformer_mae import TimeSformerEncoder
from classifier import (
    ThresholdClassifier,
    load_pretrained_encoder,
    compute_metrics,
    extract_embeddings,
    freeze_encoder,
    get_lr_scheduler,
    EarlyStopping
)
from dataset import create_dataloaders

print("✓ Imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

✓ Imports successful
PyTorch version: 2.10.0.dev20251029+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


## Configuration

In [2]:
# =============================================================================
# CONFIGURATION - ADJUST THESE!
# =============================================================================

# Paths (YOUR LOCAL PATHS)
DATA_ROOT = r"C:\\Users\\shrua\\OneDrive\\Desktop\\threshold project\\threshold\\data"
PRETRAINED_ENCODER = r"C:\\Users\\shrua\\OneDrive\\Desktop\\threshold project\\threshold\\models\\output\\timesformer_encoder_pretrained.pt"
OUTPUT_DIR = r"C:\\Users\\shrua\\OneDrive\\Desktop\\threshold project\\threshold\\models\\output_classifier"

# Model size (must match pretraining!)
MODEL_SIZE = 'default'  # 'small' or 'default' - MUST MATCH MAE TRAINING!

# Training hyperparameters
NUM_EPOCHS = 50  # Start with 50, can increase if needed
BATCH_SIZE = 16  # Can increase if you have GPU memory
LEARNING_RATE = 1e-4  # Lower than MAE (fine-tuning)
WEIGHT_DECAY = 0.01
TRAIN_SPLIT = 0.85

# Fine-tuning strategy
FREEZE_ENCODER_EPOCHS = 10  # Train only head for first N epochs
USE_EARLY_STOPPING = True
PATIENCE = 15  # Early stopping patience

# Hardware
NUM_WORKERS = 0  # Keep 0 for Windows
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Typology names
TYPOLOGY_NAMES = ['t1', 't2', 't3', 't4', 't5', 't6', 't7', 't8']

print("Configuration:")
print(f"  Pretrained encoder: {PRETRAINED_ENCODER}")
print(f"  Model size: {MODEL_SIZE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Device: {DEVICE}")
print(f"  Freeze encoder for first {FREEZE_ENCODER_EPOCHS} epochs")

Configuration:
  Pretrained encoder: C:\\Users\\shrua\\OneDrive\\Desktop\\threshold project\\threshold\\models\\output\\timesformer_encoder_pretrained.pt
  Model size: default
  Epochs: 50
  Batch size: 16
  Learning rate: 0.0001
  Device: cuda
  Freeze encoder for first 10 epochs


In [3]:
# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'checkpoints'), exist_ok=True)
print(f"✓ Output directory: {OUTPUT_DIR}")

✓ Output directory: C:\\Users\\shrua\\OneDrive\\Desktop\\threshold project\\threshold\\models\\output_classifier


## Load Data

In [4]:
print("Loading dataset...")

train_loader, val_loader = create_dataloaders(
    data_root=DATA_ROOT,
    batch_size=BATCH_SIZE,
    train_split=TRAIN_SPLIT,
    num_workers=NUM_WORKERS,
)

print(f"✓ Data loaded")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Train samples: {len(train_loader.dataset)}")
print(f"  Val samples: {len(val_loader.dataset)}")

Loading dataset...
Found 40 threshold CSV files
Loaded 10474 threshold sequences from 40 files
Train size: 8902, Val size: 1572
✓ Data loaded
  Train batches: 557
  Val batches: 99
  Train samples: 8902
  Val samples: 1572


## Load Pretrained Encoder & Create Classifier

In [5]:
# Get config (must match pretraining!)
if MODEL_SIZE == 'small':
    config = get_small_config()
    print("Using SMALL config (must match MAE training!)")
else:
    config = get_mae_config()
    print("Using DEFAULT config (must match MAE training!)")

# Load pretrained encoder
encoder = load_pretrained_encoder(PRETRAINED_ENCODER, config)

# Create classifier
model = ThresholdClassifier(
    encoder=encoder,
    num_classes=8,
    hidden_size=config.hidden_size,
    dropout=0.1
)
model = model.to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
encoder_params = sum(p.numel() for p in model.encoder.parameters())
head_params = sum(p.numel() for p in model.classifier.parameters())

print(f"\n✓ Model created")
print(f"  Total parameters: {total_params:,}")
print(f"  Encoder parameters: {encoder_params:,} (pretrained)")
print(f"  Classification head: {head_params:,} (random init)")

Configuration:
  Image size: 32x64
  Patch size: 4x4
  Patches per frame: 128 (8x16)
  Total frames: 7
  Total patches: 896
  Visible patches (~10%): ~89
  Masked patches (~90%): ~806
Using DEFAULT config (must match MAE training!)
✓ Loaded pretrained encoder from: C:\\Users\\shrua\\OneDrive\\Desktop\\threshold project\\threshold\\models\\output\\timesformer_encoder_pretrained.pt

✓ Model created
  Total parameters: 14,563,208
  Encoder parameters: 14,559,360 (pretrained)
  Classification head: 3,848 (random init)


## Setup Training

In [ ]:
# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

# Learning rate scheduler with warmup
scheduler = get_lr_scheduler(optimizer, NUM_EPOCHS, warmup_epochs=5)

# Early stopping
if USE_EARLY_STOPPING:
    early_stopping = EarlyStopping(pATIENCE=PATIENCE, mode='max')  # max for accuracy
    print(f"Early stopping: patience={PATIENCE} epochs")

print("✓ Optimizer and scheduler created")

In [ ]:
# Training history
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'learning_rate': [],
}

best_val_acc = 0.0

## Training Functions

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device, epoch):
    """
    Train for one epoch.
    """
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    pbar = tqdm(dataloader, desc=f"Epoch {epoch} [Train]")
    
    for batch in pbar:
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)
        
        # Forward pass
        logits = model(pixel_values)
        loss = criterion(logits, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Accumulate metrics
        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        # Update progress bar
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_loss = total_loss / len(dataloader)
    accuracy = np.mean(np.array(all_preds) == np.array(all_labels))
    
    return avg_loss, accuracy


def validate(model, dataloader, criterion, device, epoch):
    """
    Validate the model.
    """
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        pbar = tqdm(dataloader, desc=f"Epoch {epoch} [Val]")
        
        for batch in pbar:
            pixel_values = batch['pixel_values'].to(device)
            labels = batch['labels'].to(device)
            
            logits = model(pixel_values)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_loss = total_loss / len(dataloader)
    accuracy = np.mean(np.array(all_preds) == np.array(all_labels))
    
    return avg_loss, accuracy, all_preds, all_labels


def save_checkpoint(model, optimizer, epoch, val_acc, filename):
    """
    Save model checkpoint.
    """
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_acc': val_acc,
    }
    torch.save(checkpoint, filename)
    print(f"  💾 Saved: {os.path.basename(filename)}")


print("✓ Training functions defined")

## 🚀 START TRAINING

In [ ]:
print("="*80)
print(f"STARTING CLASSIFICATION FINE-TUNING: {NUM_EPOCHS} epochs")
print("="*80)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    
    # Freeze/unfreeze encoder
    if epoch == 0:
        freeze_encoder(model, freeze=True)
    elif epoch == FREEZE_ENCODER_EPOCHS:
        freeze_encoder(model, freeze=False)
    
    # Train
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, DEVICE, epoch
    )
    
    # Validate
    val_loss, val_acc, val_preds, val_labels = validate(
        model, val_loader, criterion, DEVICE, epoch
    )
    
    # Update scheduler
    scheduler.step()
    
    # Record history
    epoch_time = time.time() - epoch_start
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['learning_rate'].append(optimizer.param_groups[0]['lr'])
    
    # Print epoch summary
    print(f"\nEpoch {epoch}/{NUM_EPOCHS-1}")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    print(f"  LR: {optimizer.param_groups[0]['lr']:.6f} | Time: {epoch_time:.1f}s")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_path = os.path.join(OUTPUT_DIR, 'checkpoints', 'best_classifier.pt')
        save_checkpoint(model, optimizer, epoch, val_acc, best_path)
        print(f"  ⭐ New best model! (val_acc: {val_acc:.4f})")
    
    # Early stopping
    if USE_EARLY_STOPPING:
        if early_stopping(val_acc):
            print(f"\n⚠️ Early stopping triggered at epoch {epoch}")
            print(f"   No improvement for {PATIENCE} epochs")
            break
    
    print("-" * 80)

print("\n" + "="*80)
print("TRAINING COMPLETE!")
print("="*80)
print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Best validation accuracy: {best_val_acc:.4f}")

## Save Final Model

In [ ]:
# Save final model
final_path = os.path.join(OUTPUT_DIR, 'checkpoints', 'final_classifier.pt')
save_checkpoint(model, optimizer, NUM_EPOCHS - 1, history['val_acc'][-1], final_path)

print(f"\n✓ Final classifier saved")
print(f"  Best model: {os.path.join(OUTPUT_DIR, 'checkpoints', 'best_classifier.pt')}")
print(f"  Final model: {final_path}")

## Visualize Training Results

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

epochs_range = range(len(history['train_loss']))

# Plot 1: Loss curves
axes[0, 0].plot(epochs_range, history['train_loss'], 'b-', label='Train', linewidth=2)
axes[0, 0].plot(epochs_range, history['val_loss'], 'r-', label='Val', linewidth=2)
axes[0, 0].set_xlabel('Epoch', fontsize=11)
axes[0, 0].set_ylabel('Loss', fontsize=11)
axes[0, 0].set_title('Training & Validation Loss', fontsize=13, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Accuracy curves
axes[0, 1].plot(epochs_range, history['train_acc'], 'b-', label='Train', linewidth=2)
axes[0, 1].plot(epochs_range, history['val_acc'], 'r-', label='Val', linewidth=2)
axes[0, 1].set_xlabel('Epoch', fontsize=11)
axes[0, 1].set_ylabel('Accuracy', fontsize=11)
axes[0, 1].set_title('Training & Validation Accuracy', fontsize=13, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_ylim([0, 1.05])

# Plot 3: Learning rate
axes[1, 0].plot(epochs_range, history['learning_rate'], 'g-', linewidth=2)
axes[1, 0].set_xlabel('Epoch', fontsize=11)
axes[1, 0].set_ylabel('Learning Rate', fontsize=11)
axes[1, 0].set_title('Learning Rate Schedule', fontsize=13, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_yscale('log')

# Plot 4: Summary statistics
axes[1, 1].axis('off')
summary_text = f"""
TRAINING SUMMARY

Final Results:
  Train Acc: {history['train_acc'][-1]:.4f}
  Val Acc:   {history['val_acc'][-1]:.4f}
  Best Val:  {best_val_acc:.4f}

Improvement:
  Initial:  {history['val_acc'][0]:.4f}
  Final:    {history['val_acc'][-1]:.4f}
  Gain:     {(history['val_acc'][-1] - history['val_acc'][0]):.4f}

Epochs Trained: {len(epochs_range)}
"""
axes[1, 1].text(0.1, 0.5, summary_text, fontsize=11, family='monospace',
               verticalalignment='center')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Training curves saved")

## Confusion Matrix

In [ ]:
# Compute metrics on validation set
metrics = compute_metrics(val_preds, val_labels, TYPOLOGY_NAMES)

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(10, 8))

conf_matrix = metrics['confusion_matrix']
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', 
            xticklabels=TYPOLOGY_NAMES, yticklabels=TYPOLOGY_NAMES,
            ax=ax, cbar_kws={'label': 'Count'})

ax.set_xlabel('Predicted', fontsize=12, fontweight='bold')
ax.set_ylabel('True', fontsize=12, fontweight='bold')
ax.set_title('Confusion Matrix (Validation Set)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\nPer-Class Accuracy:")
for i, name in enumerate(TYPOLOGY_NAMES):
    print(f"  {name}: {metrics['per_class_accuracy'][i]:.4f}")

## 🎯 Extract Embeddings & Visualize Clustering

**This is what you've been waiting for!**

After fine-tuning, embeddings should cluster by typology.

In [ ]:
# Load best model
best_checkpoint = torch.load(os.path.join(OUTPUT_DIR, 'checkpoints', 'best_classifier.pt'))
model.load_state_dict(best_checkpoint['model_state_dict'])
print(f"✓ Loaded best model (epoch {best_checkpoint['epoch']}, val_acc: {best_checkpoint['val_acc']:.4f})")

# Extract embeddings from validation set
print("\nExtracting embeddings from validation set...")
embeddings, labels, predictions = extract_embeddings(model, val_loader, DEVICE)

print(f"✓ Extracted embeddings")
print(f"  Shape: {embeddings.shape}")
print(f"  Accuracy: {(predictions == labels).mean():.4f}")

### t-SNE Visualization

In [ ]:
# Compute t-SNE
print("Computing t-SNE (this may take a minute)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
embeddings_2d = tsne.fit_transform(embeddings)

print("✓ t-SNE complete")

# Plot t-SNE
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Plot 1: Colored by ground truth
for i, name in enumerate(TYPOLOGY_NAMES):
    mask = labels == i
    axes[0].scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1], 
                   label=name, alpha=0.6, s=30)

axes[0].set_xlabel('t-SNE Dimension 1', fontsize=12)
axes[0].set_ylabel('t-SNE Dimension 2', fontsize=12)
axes[0].set_title('t-SNE: Colored by Ground Truth', fontsize=14, fontweight='bold')
axes[0].legend(loc='best', fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot 2: Colored by predictions
for i, name in enumerate(TYPOLOGY_NAMES):
    mask = predictions == i
    axes[1].scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1], 
                   label=name, alpha=0.6, s=30)

axes[1].set_xlabel('t-SNE Dimension 1', fontsize=12)
axes[1].set_ylabel('t-SNE Dimension 2', fontsize=12)
axes[1].set_title('t-SNE: Colored by Predictions', fontsize=14, fontweight='bold')
axes[1].legend(loc='best', fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'tsne_clustering.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ t-SNE visualization complete!")
print("\nIf you see clusters, fine-tuning worked!")
print("Compare this to MAE embeddings - should be MUCH better!")

### PCA Visualization

In [ ]:
# Compute PCA
print("Computing PCA...")
pca = PCA(n_components=2)
embeddings_pca = pca.fit_transform(embeddings)

print(f"✓ PCA complete")
print(f"  Explained variance: {pca.explained_variance_ratio_[0]:.3f}, {pca.explained_variance_ratio_[1]:.3f}")
print(f"  Total: {pca.explained_variance_ratio_.sum():.3f}")

# Plot PCA
fig, ax = plt.subplots(figsize=(10, 8))

for i, name in enumerate(TYPOLOGY_NAMES):
    mask = labels == i
    ax.scatter(embeddings_pca[mask, 0], embeddings_pca[mask, 1], 
              label=name, alpha=0.6, s=30)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)', fontsize=12)
ax.set_title('PCA: Embeddings Colored by Ground Truth', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'pca_clustering.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ PCA visualization complete!")

## Summary & Next Steps

In [ ]:
print("="*80)
print("PHASE 2: CLASSIFICATION FINE-TUNING - COMPLETE!")
print("="*80)

print(f"\n📊 Final Results:")
print(f"  Best Validation Accuracy: {best_val_acc:.4f}")
print(f"  Final Validation Accuracy: {history['val_acc'][-1]:.4f}")

print(f"\n📁 Saved Files:")
print(f"  Best model: {os.path.join(OUTPUT_DIR, 'checkpoints', 'best_classifier.pt')}")
print(f"  Training curves: {os.path.join(OUTPUT_DIR, 'training_curves.png')}")
print(f"  Confusion matrix: {os.path.join(OUTPUT_DIR, 'confusion_matrix.png')}")
print(f"  t-SNE clustering: {os.path.join(OUTPUT_DIR, 'tsne_clustering.png')}")
print(f"  PCA clustering: {os.path.join(OUTPUT_DIR, 'pca_clustering.png')}")

print(f"\n✅ Key Achievement:")
print(f"  Embeddings NOW cluster by typology (check t-SNE plots!)")
print(f"  This is what you wanted - compare to MAE embeddings!")

print(f"\n⏭️ Next Steps (Phase 3):")
print(f"  1. Attention analysis - see what model focuses on")
print(f"  2. Embedding interpolation - explore hybrid thresholds")
print(f"  3. Apply to real buildings")

print("="*80)